In [19]:
import networkx as nx
from pyvis.network import Network
import os

def visualize_interactive_with_legend(file_path, output_file="graph_legend.html"):
    """
    Generates an interactive PyVis graph and injects a custom HTML Legend 
    to explain what each color means.
    """
    if not os.path.exists(file_path):
        print(f"Error: File not found at {file_path}")
        return

    # 1. Load the Graph
    print(f"Loading {file_path}...")
    try:
        G = nx.read_graphml(file_path)
    except Exception as e:
        print(f"Error parsing GraphML: {e}")
        return

    # 2. Define a Color Palette manually
    # We map colors to entity types so we can build a consistent legend
    # You can add more colors to this list if you have many entity types
    hex_colors = [
        "#FF6666", # Red (e.g., Person)
        "#66FF66", # Green (e.g., Organization)
        "#66CCFF", # Blue (e.g., Technology)
        "#FFFF66", # Yellow (e.g., Game)
        "#FFCC66", # Orange
        "#CC99FF", # Purple
        "#FF99CC", # Pink
        "#99CC99"  # Sage
    ]
    
    # Find unique entity types
    entity_types = set()
    for _, data in G.nodes(data=True):
        raw_type = data.get('entity_type', 'Unknown')
        clean_type = str(raw_type).replace('"', '')
        entity_types.add(clean_type)
    
    # Map each type to a color
    type_to_color = {}
    sorted_types = sorted(list(entity_types))
    for i, etype in enumerate(sorted_types):
        # Cycle through colors if we have more types than colors
        color = hex_colors[i % len(hex_colors)]
        type_to_color[etype] = color

    # 3. Initialize PyVis
    net = Network(height="750px", width="100%", bgcolor="#222222", font_color="white", select_menu=False)

    # 4. Add Nodes manually to apply our Specific Colors
    for node, data in G.nodes(data=True):
        # Get Type
        raw_type = data.get('entity_type', 'Unknown')
        clean_type = str(raw_type).replace('"', '')
        
        # Get Color
        assigned_color = type_to_color.get(clean_type, "#cccccc")
        
        # Clean Label
        label_text = str(node).replace('"', '')
        
        # Add to PyVis
        # We use title=... to show description on hover
        tooltip = data.get('description', clean_type)
        net.add_node(node, label=label_text, title=tooltip, color=assigned_color)

    # 5. Add Edges (if any existed, though your file has none)
    for source, target in G.edges():
        net.add_edge(source, target)


    net.force_atlas_2based(
        gravity=-50,
        central_gravity=0.01,
        spring_length=100,
        spring_strength=0.08,
        damping=0.9,
        overlap=0
    )


    # 6. Enable Physics
    net.toggle_physics(True)
    # net.show_buttons(filter_=['physics'])

    # 7. Save the initial HTML
    net.save_graph(output_file)


    # ============================================================
    # 8. THE MAGIC TRICK: Inject HTML Legend into the generated file
    # ============================================================
    
    # Create the HTML string for the legend
    legend_items = ""
    for etype, color in type_to_color.items():
        legend_items += f"""
        <div style="display: flex; align-items: center; margin-bottom: 5px;">
            <div style="width: 15px; height: 15px; background-color: {color}; margin-right: 8px; border: 1px solid #fff;"></div>
            <span style="color: white; font-family: sans-serif; font-size: 14px;">{etype}</span>
        </div>
        """

    legend_html = f"""
    <div id="custom-legend" style="
        position: absolute;
        top: 10px;
        right: 10px;
        background-color: rgba(0, 0, 0, 0.7);
        padding: 15px;
        border-radius: 8px;
        border: 1px solid #555;
        z-index: 1000;
        min-width: 150px;
    ">
        <h3 style="color: white; margin-top: 0; font-family: sans-serif; border-bottom: 1px solid #555; padding-bottom: 5px;">Entity Legend</h3>
        {legend_items}
    </div>
    """

    # Read the generated file
    with open(output_file, "r", encoding='utf-8') as f:
        content = f.read()

    # Insert the legend right before the closing body tag
    new_content = content.replace("</body>", f"{legend_html}</body>")

    # Write it back
    with open(output_file, "w", encoding='utf-8') as f:
        f.write(new_content)

    print(f"Success! Interactive graph with Legend saved to: {os.path.abspath(output_file)}")

# --- USAGE ---
path = "/home/gatv-projects/Desktop/project/playground/videorag_cache_2025-11-20-12:09:09/graph_chunk_entity_relation.graphml"  # Replace with your path
visualize_interactive_with_legend(path)

Loading /home/gatv-projects/Desktop/project/playground/videorag_cache_2025-11-20-12:09:09/graph_chunk_entity_relation.graphml...
Success! Interactive graph with Legend saved to: /home/gatv-projects/Desktop/project/playground/knowledge_graph_build/graph_legend.html
